# Import

In [13]:
import string
import IPython
from IPython.display import Audio
import torch
import os

import torchaudio

from TTS.tts.utils.synthesis import synthesis
from TTS.utils.audio import AudioProcessor
from TTS.tts.models import setup_model
from TTS.config import load_config
from TTS.tts.models.vits import *
from TTS.tts.utils.speakers import SpeakerManager
from TTS.utils.vad import get_vad_model_and_utils, remove_silence, resample_wav, read_audio

from pydub import AudioSegment

# Define Parameter and Constant

In [14]:
OUT_PATH = 'output'
BASE_MODEL_PATH = './model/tsync2_cv_1'
REFERENCE_FILENAME = 'ron.wav'

# model vars 
MODEL_PATH = os.path.join(BASE_MODEL_PATH, 'checkpoint_453866.pth')
CONFIG_PATH = os.path.join(BASE_MODEL_PATH, 'config.json')
TTS_LANGUAGES = os.path.join(BASE_MODEL_PATH, 'language_ids.json')
USE_CUDA = torch.cuda.is_available()
REFERENCE_PATH = os.path.join("./reference_voice", REFERENCE_FILENAME.strip().strip('./').strip('/'))
REFERENCE_WAV_PATH = os.path.join("./reference_voice", "wav", REFERENCE_FILENAME.strip().strip('./').strip('/').split(".")[0] + ".wav")
SPEAKER_FOLDER = REFERENCE_PATH.split("/")[-1].split(".")[0]

model_name = MODEL_PATH.rstrip('.pth').split('/')[-1]

# Setup Model and Config

In [15]:
# load the config
C = load_config(CONFIG_PATH)

# load the audio processor
ap = AudioProcessor(**C.audio)

# override config
C["speakers_file"] = None
C["d_vector_file"] = []
C["language_ids_file"] = TTS_LANGUAGES

C["model_args"]["speakers_file"] = None
C["model_args"]["d_vector_file"] = []
C["model_args"]["language_ids_file"] = TTS_LANGUAGES

C.model_args['use_speaker_encoder_as_loss'] = False

model = setup_model(C)
cp = torch.load(MODEL_PATH, map_location=torch.device('cpu'))

# remove speaker encoder
model_weights = cp['model'].copy()
for key in list(model_weights.keys()):
  if "speaker_encoder" in key:
    del model_weights[key]

model.load_state_dict(model_weights)

model.eval()

if USE_CUDA:
  model = model.cuda()

# synthesize voice
use_griffin_lim = False

 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > 

# Setup Language

In [16]:
# Select language
language_id = 0
language_name_to_id = model.language_manager.name_to_id
language_id_to_name = {v: k for k, v in language_name_to_id.items()}
print(f"Language ID: {language_id}, Language Name: {language_id_to_name[language_id]}")

Language ID: 0, Language Name: th


# Process reference audio file

In [17]:
# Check if refernec voice is in wav format
current_ref_extension = REFERENCE_PATH.split(".")[-1].lower()

# Convert to wav if not or never converted
if current_ref_extension != "wav":
    if not os.path.exists(REFERENCE_WAV_PATH):
        audio: AudioSegment = AudioSegment.from_file(REFERENCE_PATH, format=current_ref_extension)
        audio.export(REFERENCE_WAV_PATH, format="wav")
    REFERENCE_PATH = REFERENCE_WAV_PATH

In [18]:
# Resmapling reference voice if needed
ref_wav, current_sr = read_audio(REFERENCE_PATH)
if current_sr != C.audio['sample_rate']:
    print('Resampling reference audio...')
    resamapled_ref_wav = resample_wav(ref_wav, current_sr, C.audio['sample_rate'])
    torchaudio.save(REFERENCE_PATH, resamapled_ref_wav[None, :], C.audio['sample_rate'])
else:
    print('This audio is already in the correct sample rate')

This audio is already in the correct sample rate


In [19]:
# trim silence at the beginning and end of the audio
model_and_utils = get_vad_model_and_utils(use_cuda=USE_CUDA, use_onnx=False)

output_path, is_speech = remove_silence(
  model_and_utils,
  REFERENCE_PATH,
  REFERENCE_PATH,
  trim_just_beginning_and_end=True,
  use_cuda=USE_CUDA
)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /Users/jackkahod/.cache/torch/hub/master.zip


In [20]:
# normalize the reference audio with rms to -27dB
!ffmpeg-normalize $REFERENCE_PATH -nt rms -t=-27 -o $REFERENCE_PATH -ar 16000 -f

In [21]:
SE_speaker_manager = SpeakerManager(encoder_model_path=C["model_args"]["speaker_encoder_model_path"], encoder_config_path=C["model_args"]["speaker_encoder_config_path"], use_cuda=USE_CUDA)
reference_emb = SE_speaker_manager.compute_embedding_from_clip(REFERENCE_PATH)

 > Model fully restored. 
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:64
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:512
 | > power:1.5
 | > preemphasis:0.97
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:False
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:False
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:True
 | > db_level:-27.0
 | > stats_path:None
 | > base:10
 | > hop_length:160
 | > win_length:400


# Setup duration predictor

In [22]:
model.length_scale = 0.8 # scaler for the duration predictor. The larger it is, the slower the speech.
model.inference_noise_scale = 0.2 # defines the noise variance applied to the random z vector at inference.
model.inference_noise_scale_dp = 0.2 # defines the noise variance applied to the duration predictor z vector at inference.

# Inference

In [23]:
text = "สวัสดีฉันชื่อรอน รอน วิสลี่"
print(f" > text: {text} with sampling rate: {ap.sample_rate}")

wav, alignment, _, _ = synthesis(
                    model = model,
                    text = text,
                    CONFIG = C,
                    use_cuda = USE_CUDA,
                    d_vector = reference_emb,
                    style_wav = None,
                    language_id = language_id,
                    use_griffin_lim = True,
                    do_trim_silence = False,
                ).values()
print("Audio Generated")
IPython.display.display(Audio(wav, rate=ap.sample_rate))

 > text: สวัสดีฉันชื่อรอน รอน วิสลี่ with sampling rate: 16000
Audio Generated


# Save to Folder

In [24]:
file_name = text.replace(" ", "_")
file_name = model_name + '_' + file_name.translate(str.maketrans('', '', string.punctuation.replace('_', ''))) + '.wav'
out_path = os.path.join(OUT_PATH, f"{SPEAKER_FOLDER}/{file_name}")

print(f" > Saving output to {out_path}")

os.makedirs(f"{OUT_PATH}/{SPEAKER_FOLDER}", exist_ok=True)
ap.save_wav(wav, out_path)

 > Saving output to output/ron/checkpoint_453866_สวัสดีฉันชื่อรอน_รอน_วิสลี่.wav
